# RFdiffusion on Colab (v2)

Protein backbone generation. Run steps 1-6 once (~12 min), then use step 7 for designs.

**First: Runtime -> Change runtime type -> T4 GPU.**

<small>**Using Run all?** Two things need you: step 2 waits for a Google Drive
authorisation popup, and step 7's fields must already say what you want built - its
defaults make a 100-residue monomer and ignore any file in Drive. Set those two up and
the rest, including saving back to Drive, runs unattended. The setup steps now raise on
a bad environment, so a Run all stops at the real cause instead of cascading.</small>

<small>Runs RFdiffusion inside its own Python 3.10 environment, called as a program rather
than imported, because Colab's own Python is 3.13 and RFdiffusion's dependencies have no
3.13 builds. Double-click a step title to see its code.</small>

In [ ]:
#@title 1. Check the GPU
#@markdown Must show a GPU. If not, fix the runtime type before going further.

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

In [ ]:
#@title 2. Mount your Google Drive
#@markdown Gives the notebook a folder on your Drive, so you can feed it a target
#@markdown structure you keep there and have the designs land back there too.
#@markdown A browser popup will ask you to authorise access.
drive_folder = "RFdiffusion" #@param {type:"string"}

import os
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT   = f"/content/drive/MyDrive/{drive_folder}"
DRIVE_INPUTS = f"{DRIVE_ROOT}/inputs"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/outputs"
os.makedirs(DRIVE_INPUTS, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

print(f"Drive mounted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Put target structures in:  My Drive/{drive_folder}/inputs")
print(f"Designs will be saved to:  My Drive/{drive_folder}/outputs\n")

found = sorted(f for f in os.listdir(DRIVE_INPUTS)
               if f.lower().endswith((".pdb", ".cif", ".ent")))
if found:
    print("Structures already in your inputs folder - paste one of these names")
    print("into `input_pdb` in step 7:")
    for f in found:
        print("   ", f)
else:
    print("inputs folder is empty for now. Drop a .pdb there (the Files pane on the")
    print("left refreshes with the folder icon) and re-run this cell to list it.")


In [ ]:
#@title 3. Install Python 3.10
#@markdown Sits alongside Colab's own Python without disturbing it.

!apt-get update -qq
!apt-get install -y -qq python3.10 python3.10-venv python3.10-dev
!python3.10 --version

In [ ]:
#@title 4. Build the environment (~6 min, ~3 GB)
#@markdown Installs PyTorch 2.0.1 / CUDA 11.8, DGL 1.1.3, the CUDA 11 runtime libraries
#@markdown DGL loads by hand, and NumPy 1.x. Let it finish - do not interrupt it or
#@markdown re-run it part way through.
#@markdown
#@markdown Ends by checking itself and **raising** if anything is wrong, so a Run all
#@markdown stops here rather than failing later for a misleading reason.

import os, glob

ENV = "/content/rfenv"
PY  = f"{ENV}/bin/python"
PIP = f"{ENV}/bin/pip"
SP  = f"{ENV}/lib/python3.10/site-packages"

# A half-built environment is worse than none: rebuild rather than install on top.
if not os.path.isfile(PY):
    print("creating environment...")
    os.system(f"rm -rf {ENV}")
    os.system(f"python3.10 -m venv {ENV}")
assert os.path.isfile(PY), "venv creation failed - check step 3's output"

!{PIP} install -q --upgrade pip setuptools wheel
!{PIP} install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118

# DGL opens the CUDA 11 libraries itself through the system loader, so they must
# exist as real files regardless of what PyTorch bundles privately.
!{PIP} install -q nvidia-cublas-cu11 nvidia-cusparse-cu11 nvidia-cusolver-cu11 nvidia-cufft-cu11 nvidia-curand-cu11 nvidia-cuda-runtime-cu11 nvidia-cuda-nvrtc-cu11 nvidia-cudnn-cu11
!{PIP} install -q dgl==1.1.3 -f https://data.dgl.ai/wheels/cu118/repo.html
!{PIP} install -q hydra-core omegaconf icecream e3nn==0.5.1 opt_einsum opt_einsum_fx biopython decorator pyrsistent pynvml jedi tqdm

# Pinned last so nothing drags NumPy 2 back in. PyTorch 2.0.1 was compiled against
# NumPy 1.x and fails to initialise against 2.x.
!{PIP} install -q "numpy==1.26.4" "scipy==1.11.4" "pandas==2.0.3"

# Directories holding the CUDA shared libraries DGL needs to find
LDPATH = ":".join(sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"])

print("\n--- environment check ---")
hit = glob.glob(f"{SP}/**/libcusparse.so.11", recursive=True)
print("libcusparse.so.11:", hit[0] if hit else "*** NOT FOUND ***")

lines = [
    "import sys, torch, dgl, numpy",
    "print('python:', sys.version.split()[0])",
    "print('torch :', torch.__version__)",
    "print('dgl   :', dgl.__version__)",
    "print('numpy :', numpy.__version__)",
    "print('cuda  :', torch.cuda.is_available())",
    "print('gpu   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')",
    "print('torch<->numpy:', torch.tensor([1.0]).numpy().sum())",
]
with open("/content/_check.py", "w") as f:
    f.write("\n".join(lines))

import subprocess
_chk = subprocess.run([PY, "/content/_check.py"],
                      env=dict(os.environ, DGLBACKEND="pytorch", LD_LIBRARY_PATH=LDPATH),
                      capture_output=True, text=True)
print(_chk.stdout + _chk.stderr)

# Verified rather than left for the eye, so an unattended Run all halts on the real
# cause instead of carrying a broken environment into the steps below.
_vals = dict(l.split(":", 1) for l in _chk.stdout.splitlines() if ":" in l)
_vals = {k.strip(): v.strip() for k, v in _vals.items()}
_bad = []
if _chk.returncode:
    _bad.append("the check script itself failed - see its output above")
if not hit:
    _bad.append("libcusparse.so.11 is missing: the nvidia-*-cu11 installs failed, "
                "usually from a full disk")
if not _vals.get("numpy", "").startswith("1."):
    _bad.append(f"numpy is {_vals.get('numpy', '?')}, must be 1.x - something "
                "reinstalled NumPy 2 over the pin")
if _vals.get("cuda") != "True":
    _bad.append("torch cannot see the GPU - check the runtime type (step 1)")
if not _vals.get("dgl", "").startswith("1.1"):
    _bad.append(f"dgl is {_vals.get('dgl', 'missing or failed to import')}, must be 1.1.x")

if _bad:
    print("\n*** ENVIRONMENT IS NOT USABLE ***")
    for _b in _bad:
        print("  - " + _b)
    raise RuntimeError(
        "Environment check failed (" + str(len(_bad)) + " problem(s), listed above). "
        "Do NOT re-run this cell on top of the failure: Runtime > Disconnect and "
        "delete runtime, then start again from step 1.")

print("\nEnvironment OK.")

**Check step 4's output before continuing.** You want `numpy: 1.26.4`, `dgl: 1.1.3`,
`cuda: True`, a GPU name, and a `torch<->numpy` line printing `1.0`. If DGL raised an
error or NumPy reads 2.x, stop here - nothing below will work, and the errors it produces
will point somewhere misleading.

In [ ]:
#@title 5. Download RFdiffusion and model weights (~1 GB)

import os
%cd /content
!apt-get install -y -qq aria2

if not os.path.isdir("/content/RFdiffusion"):
    !git clone -q https://github.com/sokrypton/RFdiffusion.git

!mkdir -p /content/RFdiffusion/models

# No schedules download here. That file (files.ipd.uw.edu/krypton/schedules.zip)
# was removed from the server and now returns 404. It only ever held a cache:
# RFdiffusion computes the IGSO3 schedule itself when the folder is absent and
# saves it, so the first design run takes a few extra minutes and later ones do not.

if not os.path.isfile("/content/RFdiffusion/models/Base_ckpt.pt"):
    !aria2c -x 16 -d /content/RFdiffusion/models http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt
if not os.path.isfile("/content/RFdiffusion/models/Complex_base_ckpt.pt"):
    !aria2c -x 16 -d /content/RFdiffusion/models http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt

!ls -lh /content/RFdiffusion/models/

In [ ]:
#@title 6. Compile SE3Transformer (~3 min)
#@markdown RFdiffusion's equivariant attention layers. Ends by testing the exact import
#@markdown chain that fails in the standard notebook. If it prints OK, setup is done.
#@markdown
#@markdown RFdiffusion itself is deliberately *not* pip-installed: this fork ships no
#@markdown setup.py at its root, so `pip install -e .` fails. Its modules are found by
#@markdown running from inside its own directory instead.

import os, glob
ENV = "/content/rfenv"; PY = f"{ENV}/bin/python"; PIP = f"{ENV}/bin/pip"
SP  = f"{ENV}/lib/python3.10/site-packages"
LDPATH = ":".join(sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"])

!cd /content/RFdiffusion/env/SE3Transformer && {PIP} install -q --no-deps .

import subprocess
_imp = subprocess.run([PY, "-c", "import inference.utils"], cwd="/content/RFdiffusion",
                      env=dict(os.environ, DGLBACKEND="pytorch", LD_LIBRARY_PATH=LDPATH),
                      capture_output=True, text=True)
print(_imp.stdout + _imp.stderr)

# A marker file, so the design step can tell setup finished rather than assuming it
_MARK = "/content/.rf_setup_ok"
if _imp.returncode:
    if os.path.isfile(_MARK):
        os.remove(_MARK)
    raise RuntimeError(
        "RFdiffusion still does not import - see the traceback above. This is the "
        "failure the whole setup exists to avoid, so nothing below will work. "
        "Runtime > Disconnect and delete runtime, then start again from step 1.")

for _w in ("Base_ckpt.pt", "Complex_base_ckpt.pt"):
    _p = f"/content/RFdiffusion/models/{_w}"
    if not os.path.isfile(_p) or os.path.getsize(_p) < 10_000_000:
        raise RuntimeError(f"{_w} is missing or truncated - re-run step 5.")

open(_MARK, "w").write("ok")
print("RFdiffusion imports: OK - setup complete")

In [ ]:
#@title 7. Generate designs
#@markdown ### Run mode
run_mode = "Production" #@param ["Production", "Fast debug - your settings, 5 steps", "Smoke test - ignore fields, tiny monomer"]
#@markdown <small>`Production` runs exactly what the fields below say. `Fast debug` keeps
#@markdown your target, contigs and hotspots but cuts the diffusion to 5 steps and 1
#@markdown design, so a real run's settings can be checked in a fraction of the time -
#@markdown the backbones it returns are noise, not designs. `Smoke test` ignores every
#@markdown field and builds a 20-residue monomer, to prove the install works.</small>
#@markdown ---
#@markdown ### What to build
name = "test" #@param {type:"string"}
design_type = "monomer" #@param ["monomer", "binder", "motif scaffolding", "partial diffusion"]
contigs = "100-100" #@param {type:"string"}
#@markdown ### Target structure (binder / motif / partial only)
#@markdown Three ways to give one, in the order they are tried: the **name of a file in
#@markdown your Drive inputs folder** (`tgfb1_dimer.pdb`), a **full path**, or a
#@markdown **4-letter PDB code** such as `1D8R`, which is downloaded for you.
#@markdown Leave blank for a monomer.
input_pdb = "" #@param {type:"string"}
#@markdown Hotspot residues the binder should engage, e.g. `A59,A83,A91`. Strongly
#@markdown recommended for binder design - without them the binder lands anywhere.
hotspots = "" #@param {type:"string"}
#@markdown ### Sampling
iterations = 50 #@param [25, 50, 100, 150, 200] {type:"raw"}
num_designs = 1 #@param [1, 2, 4, 8, 16, 32] {type:"raw"}

#@markdown ---
#@markdown **Contig syntax**
#@markdown
#@markdown | Goal | `contigs` | `input_pdb` |
#@markdown |---|---|---|
#@markdown | Monomer, fixed length 100 | `100-100` | blank |
#@markdown | Monomer, sampled 70-100 | `70-100` | blank |
#@markdown | Binder to chain A | `A1-150/0 70-100` | target |
#@markdown | Scaffold a motif | `40/A163-181/40` | target |
#@markdown | Keep chain A, rebuild rest | `A` | target |
#@markdown
#@markdown `/0` marks a chain break. Start with 50 iterations and 1 design to confirm the
#@markdown pipeline runs before committing time to a real batch.

import subprocess, os, glob
ENV = "/content/rfenv"; PY = f"{ENV}/bin/python"
if not os.path.isfile("/content/.rf_setup_ok"):
    raise RuntimeError(
        "Setup has not completed in this runtime. Run steps 3-6 first; step 6 writes "
        "the marker this checks. If they did run, one of them failed - scroll up.")
SP  = f"{ENV}/lib/python3.10/site-packages"
LDPATH = ":".join(sorted(glob.glob(f"{SP}/nvidia/*/lib")) + [f"{SP}/torch/lib"])
os.makedirs("/content/outputs", exist_ok=True)

# --- run mode -------------------------------------------------------------
# Fast debug keeps what you are actually testing (the target, the contig string,
# the hotspots) and only shrinks the sampling. Smoke test throws all of it away
# and runs the smallest thing RFdiffusion can do, to separate "my settings are
# wrong" from "the environment is broken".
FAST = run_mode.startswith("Fast")
SMOKE = run_mode.startswith("Smoke")

if FAST:
    iterations, num_designs = 5, 1
elif SMOKE:
    name, design_type, contigs = "smoketest", "monomer", "20-20"
    input_pdb, hotspots = "", ""
    iterations, num_designs = 5, 1

if FAST or SMOKE:
    print("*" * 70)
    print(f"* {run_mode}")
    print("* 5 diffusion steps instead of a real schedule: the backbones this")
    print("* produces are NOT meaningful designs. Use Production for real work.")
    print("*" * 70, "\n")

# Resolve the target: a name in the Drive inputs folder, a path, or a PDB code
DRIVE_INPUTS = globals().get("DRIVE_INPUTS", "/content/drive/MyDrive/RFdiffusion/inputs")

pdb_path = input_pdb.strip()
if pdb_path and not os.path.isfile(pdb_path):
    from_drive = os.path.join(DRIVE_INPUTS, pdb_path)
    if os.path.isfile(from_drive):
        pdb_path = from_drive
        print(f"using target from Drive: {pdb_path}")
    elif len(pdb_path) == 4:
        dest = f"/content/{pdb_path}.pdb"
        if not os.path.isfile(dest):
            print(f"fetching {pdb_path} from the PDB...")
            os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_path}.pdb -O {dest}")
        pdb_path = dest
    else:
        raise FileNotFoundError(
            f"'{input_pdb}' is not a file, not in {DRIVE_INPUTS}, and not a 4-letter "
            "PDB code. Run step 2 to mount Drive, check the spelling, or give a full path."
        )

if design_type != "monomer" and not pdb_path:
    raise ValueError(f"design_type is '{design_type}' but input_pdb is empty")

opts = [
    f"inference.output_prefix=/content/outputs/{name}",
    f"inference.num_designs={num_designs}",
    f"contigmap.contigs=[{contigs}]",
]
opts.append(f"diffuser.partial_T={iterations}" if design_type == "partial diffusion"
            else f"diffuser.T={iterations}")
if pdb_path:
    opts.append(f"inference.input_pdb={pdb_path}")
if hotspots.strip():
    opts.append(f"ppi.hotspot_res=[{hotspots.strip()}]")

# Per-step trajectory PDBs are large and only useful for inspecting a real run.
# The flag is only passed if this RFdiffusion version actually defines it -
# hydra aborts on an unknown key rather than ignoring it.
_base_yaml = "/content/RFdiffusion/config/inference/base.yaml"
if (FAST or SMOKE) and os.path.isfile(_base_yaml):
    if "write_trajectory" in open(_base_yaml).read():
        opts.append("inference.write_trajectory=False")

cmd = [PY, "run_inference.py"] + opts
print(" ".join(cmd), "\n" + "-" * 70)

# Stream output live so progress and real errors appear as they happen
proc = subprocess.Popen(
    cmd, cwd="/content/RFdiffusion",
    env=dict(os.environ, DGLBACKEND="pytorch", LD_LIBRARY_PATH=LDPATH),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

print("-" * 70)
if proc.returncode:
    print("FAILED - see the traceback above")
elif SMOKE:
    print("Smoke test passed: the environment, the weights and the output path all work.")
    print("Anything that fails now is in your run settings, not the install.")
elif FAST:
    print("Fast debug finished: the settings are accepted and the pipeline runs.")
    print("Set run_mode back to Production for designs worth keeping.")
else:
    print("Done")
!ls -lh /content/outputs/

In [ ]:
#@title 8. View a design
design = 0 #@param {type:"integer"}
style = "cartoon" #@param ["cartoon", "stick", "sphere"]
color = "spectrum" #@param ["spectrum", "chain", "ssJmol"]

import glob, os
try:
    import py3Dmol
except ImportError:
    !pip install -q py3Dmol
    import py3Dmol

pdbs = sorted(glob.glob(f"/content/outputs/{name}_*.pdb"))
print("found:", [os.path.basename(p) for p in pdbs] or "nothing - did step 7 succeed?")

if pdbs:
    design = min(design, len(pdbs) - 1)
    print("showing:", os.path.basename(pdbs[design]))
    view = py3Dmol.view(width=760, height=520)
    view.addModel(open(pdbs[design]).read(), "pdb")
    key = "colorscheme" if color == "ssJmol" else "color"
    view.setStyle({style: {key: color}})
    view.setBackgroundColor("white")
    view.zoomTo()
    view.show()

In [ ]:
#@title 9. Save results to Drive
#@markdown Colab wipes /content when the runtime ends. Run this before closing the tab.
#@markdown Leave the field blank to use the outputs folder created in step 2.
folder = "" #@param {type:"string"}

import os
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

dest = (f"/content/drive/MyDrive/{folder}" if folder.strip()
        else globals().get("DRIVE_OUTPUT", "/content/drive/MyDrive/RFdiffusion/outputs"))
os.makedirs(dest, exist_ok=True)
!cp -r /content/outputs/* "{dest}/" 2>/dev/null
!ls -lh "{dest}"
print(f"\nSaved to Drive: {dest.replace('/content/drive/MyDrive', 'My Drive')}")


---
## If something fails

**A Run all stopped partway.** That is the environment check in step 4 or the import
check in step 6 doing its job - read the message it raised, which names what failed.
Neither is a transient error worth retrying in place; see the next paragraph.

**Never re-run a setup step on top of a partial failure.** The install is skipped because a
folder already exists, and a later step then fails for a reason unrelated to the real
problem. Instead: Runtime -> **Disconnect and delete runtime**, then start from step 1. A
plain restart keeps the files, so it keeps the problem.

**`libcusparse.so.11: cannot open shared object file`** - DGL cannot find the CUDA 11
libraries. Step 4 prints where `libcusparse.so.11` lives; if that says NOT FOUND, the
`nvidia-*-cu11` installs failed, usually from a full disk.

**`module compiled using NumPy 1.x cannot be run in NumPy 2.x`** - something reinstalled
NumPy 2. Re-run the last pip line of step 4 to pin it back to 1.26.4.

**Step 7 rejects a config key** - flag names shift between RFdiffusion versions. Run
`!cd /content/RFdiffusion && /content/rfenv/bin/python run_inference.py --help`.

**Out of memory** - shorten the contig or lower the iterations. A free T4 manages a few
hundred residues.

**Is it my settings or the install?** Set step 7's `run_mode` to **Smoke test**. It
ignores every field and diffuses a 20-residue monomer in well under a minute. If that
works the environment is fine and the problem is in the run settings; if it fails, the
problem is upstream of them and the steps above apply.

## What comes after this

RFdiffusion produces a backbone - a shape, not yet a protein. Two further steps are needed
before anything is orderable:

1. **ProteinMPNN** designs an amino acid sequence onto the backbone.
2. **AlphaFold** checks that sequence folds back into the intended shape.

Those need their own environments, so they are not bolted on here. The
[ProteinMPNN notebook](https://colab.research.google.com/github/dauparas/ProteinMPNN/blob/main/colab_notebooks/quickdemo.ipynb)
takes these PDB files directly, and
[ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)
validates the sequences it returns.

For binder design the usual filter is AlphaFold predicting the complex confidently with the
binder landing within ~2 A of the design. Expect a low hit rate - that is normal, and it is
why designs are generated in bulk.